# Bayesian Optimization for Hyperparameter Tuning
**Author:** Felipe Taha Sant'Ana  
**Scope:** GP surrogate, acquisition functions (EI/UCB/PI), practical HPO, convergence analysis

---
## Overview
Bayesian Optimization (BO) is a sample-efficient global optimization method ideal for expensive black-box functions — like ML model evaluation. This project implements BO from the ground up:

- **Gaussian Process** surrogate model with Matérn kernel
- **Acquisition functions**: Expected Improvement, UCB, Probability of Improvement
- **1D visualization** of GP fit + acquisition over iterations
- **2D benchmark** on the Branin test function
- **Practical HPO**: tuning GradientBoosting on a classification task
- **Comparison** vs Random Search and Grid Search
- **Convergence analysis**, cumulative regret, exploration-exploitation tradeoff

## Contents
1. [GP Surrogate & Acquisition Functions](#1) — 2. [BO Iterations](#2) — 3. [2D Branin Benchmark](#3)
4. [Practical HPO](#4) — 5. [Analysis](#5) — 6. [Conclusions](#6)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from itertools import product
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
COLORS = {'primary':'#8B5CF6','secondary':'#0EA5E9','accent':'#F59E0B','danger':'#EF4444',
          'success':'#10B981','dark':'#0F172A','rose':'#F43F5E','teal':'#14B8A6'}
palette = list(COLORS.values()); np.random.seed(42)
print("Libraries loaded")

In [ ]:
# ── Acquisition functions (from scratch) ────────────────────────────────
def expected_improvement(mu, sigma, best_f, xi=0.01):
    imp = mu - best_f - xi; Z = imp / (sigma + 1e-9)
    ei = imp * stats.norm.cdf(Z) + sigma * stats.norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0; return ei

def upper_confidence_bound(mu, sigma, kappa=2.0):
    return mu + kappa * sigma

def probability_improvement(mu, sigma, best_f, xi=0.01):
    Z = (mu - best_f - xi) / (sigma + 1e-9)
    pi = stats.norm.cdf(Z); pi[sigma < 1e-10] = 0.0; return pi

print("Acquisition functions: EI, UCB, PI defined")

<a id="1"></a>
## 1. GP Surrogate & Acquisition Functions (1D)

In [ ]:
def target_1d(x): return np.sin(3*x)*x + 0.5*np.cos(5*x) + 0.3*x
x_plot = np.linspace(0, 5, 500).reshape(-1, 1); y_true = target_1d(x_plot.ravel())
X_obs = np.array([0.5, 1.5, 3.0, 4.5]).reshape(-1, 1)
y_obs = target_1d(X_obs.ravel()) + np.random.normal(0, 0.1, len(X_obs))

kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=0.1, n_restarts_optimizer=5, random_state=42)
gp.fit(X_obs, y_obs)
mu, sigma = gp.predict(x_plot, return_std=True)
ei = expected_improvement(mu, sigma, y_obs.max())

fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [2, 1]})
axes[0].plot(x_plot, y_true, 'k--', linewidth=1.5, alpha=0.4, label='True Function')
axes[0].plot(x_plot, mu, linewidth=2.5, color=COLORS['primary'], label='GP Mean')
axes[0].fill_between(x_plot.ravel(), mu-2*sigma, mu+2*sigma, alpha=0.2, color=COLORS['primary'], label='±2σ')
axes[0].scatter(X_obs, y_obs, s=120, color=COLORS['danger'], edgecolor='white', linewidth=2, zorder=5, label='Observations')
axes[0].set_title('Gaussian Process Surrogate Model', fontweight='bold', fontsize=14); axes[0].legend()
axes[1].fill_between(x_plot.ravel(), 0, ei/(ei.max()+1e-9), alpha=0.3, color=COLORS['success'])
axes[1].plot(x_plot, ei/(ei.max()+1e-9), linewidth=2.5, color=COLORS['success'], label='Expected Improvement')
axes[1].axvline(x_plot[np.argmax(ei)], color=COLORS['danger'], linestyle='--', linewidth=2,
    label=f'Next sample: x={x_plot[np.argmax(ei)][0]:.2f}')
axes[1].set_title('Acquisition Function', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

<a id="2"></a>
## 2. BO Iteration Steps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
X_bo, y_bo = X_obs.copy(), y_obs.copy()
for step in range(6):
    ax = axes[step//3][step%3]
    gp_s = GaussianProcessRegressor(kernel=kernel, alpha=0.1, n_restarts_optimizer=3, random_state=42)
    gp_s.fit(X_bo, y_bo); mu_s, sigma_s = gp_s.predict(x_plot, return_std=True)
    ei_s = expected_improvement(mu_s, sigma_s, y_bo.max())
    ax.plot(x_plot, y_true, 'k--', linewidth=1, alpha=0.3)
    ax.plot(x_plot, mu_s, linewidth=2, color=COLORS['primary'])
    ax.fill_between(x_plot.ravel(), mu_s-2*sigma_s, mu_s+2*sigma_s, alpha=0.15, color=COLORS['primary'])
    ax.scatter(X_bo, y_bo, s=80, color=COLORS['danger'], edgecolor='white', zorder=5)
    nxi = np.argmax(ei_s); nx, ny = x_plot[nxi][0], target_1d(nx)+np.random.normal(0,0.1)
    ax.scatter([nx], [ny], s=150, marker='*', color=COLORS['success'], edgecolor='white', linewidth=2, zorder=6)
    ax.set_title(f'Step {step+1} (n={len(X_bo)}, best={y_bo.max():.2f})', fontweight='bold', fontsize=11)
    X_bo = np.vstack([X_bo, [[nx]]]); y_bo = np.append(y_bo, ny)
plt.suptitle('Bayesian Optimization — Iterative Acquisition', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout(); plt.show()

<a id="3"></a>
## 3. 2D Branin Benchmark

In [ ]:
def branin(x1, x2):
    a,b,c,r,s,t = 1,5.1/(4*np.pi**2),5/np.pi,6,10,1/(8*np.pi)
    return -(a*(x2-b*x1**2+c*x1-r)**2+s*(1-t)*np.cos(x1)+s)

bounds_b = np.array([[-5,10],[0,15]]); n_init, n_iter = 5, 40
X_b = np.random.uniform(bounds_b[:,0], bounds_b[:,1], size=(n_init, 2))
y_b = np.array([branin(x[0],x[1]) for x in X_b]); best_hist = [y_b.max()]
kernel_2d = ConstantKernel(1.0)*Matern(length_scale=1.0, nu=2.5)
for i in range(n_iter):
    gp2 = GaussianProcessRegressor(kernel=kernel_2d, alpha=0.01, n_restarts_optimizer=3, random_state=i)
    gp2.fit(X_b, y_b); cands = np.random.uniform(bounds_b[:,0], bounds_b[:,1], size=(1000,2))
    mu_c, sig_c = gp2.predict(cands, return_std=True); ei_c = expected_improvement(mu_c, sig_c, y_b.max())
    nx = cands[np.argmax(ei_c)]; ny = branin(nx[0],nx[1])+np.random.normal(0,0.01)
    X_b = np.vstack([X_b, nx]); y_b = np.append(y_b, ny); best_hist.append(y_b.max())
# Random baseline
Xr = np.random.uniform(bounds_b[:,0], bounds_b[:,1], size=(n_init+n_iter,2))
yr = np.array([branin(x[0],x[1]) for x in Xr])
rand_hist = [yr[:i+1].max() for i in range(len(yr))]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
x1r, x2r = np.linspace(-5,10,80), np.linspace(0,15,80)
X1, X2 = np.meshgrid(x1r, x2r)
Z = np.array([[branin(a,b) for a in x1r] for b in x2r])
c = axes[0].contourf(X1, X2, Z, levels=30, cmap='viridis', alpha=0.8)
plt.colorbar(c, ax=axes[0]); axes[0].scatter(X_b[n_init:,0], X_b[n_init:,1], s=40, color=COLORS['danger'], edgecolor='white')
bi = np.argmax(y_b); axes[0].scatter(X_b[bi,0], X_b[bi,1], s=200, marker='*', color=COLORS['accent'], edgecolor='white', zorder=6)
axes[0].set_title('BO Sampling on Branin', fontweight='bold')
axes[1].plot(best_hist, linewidth=2.5, color=COLORS['primary'], label='BO')
axes[1].plot(rand_hist, linewidth=2.5, color=COLORS['accent'], label='Random')
axes[1].axhline(-0.397, color=COLORS['success'], linestyle='--', linewidth=2, label='Global optimum')
axes[1].set_title('Convergence', fontweight='bold'); axes[1].legend(); axes[1].set_xlabel('Evaluations')
plt.tight_layout(); plt.show()

<a id="4"></a>
## 4. Practical HPO — Tuning GradientBoosting

In [ ]:
X_cls, y_cls = make_classification(n_samples=1000, n_features=15, n_informative=10,
    n_redundant=2, n_classes=2, random_state=42)
X_cls = StandardScaler().fit_transform(X_cls)

param_names = ['n_estimators','max_depth','learning_rate','min_samples_leaf','subsample']
bounds_h = np.array([[50,300],[2,12],[0.01,0.5],[5,50],[0.5,1.0]])
def denorm(x): return bounds_h[:,0]+x*(bounds_h[:,1]-bounds_h[:,0])
def eval_params(p):
    m = GradientBoostingClassifier(n_estimators=int(p[0]),max_depth=int(p[1]),
        learning_rate=p[2],min_samples_leaf=int(p[3]),subsample=p[4],random_state=42)
    return cross_val_score(m, X_cls, y_cls, cv=StratifiedKFold(2, shuffle=True, random_state=42),
        scoring='roc_auc', n_jobs=-1).mean()

# BO
n_init_h, n_iter_h = 6, 24; nt = n_init_h+n_iter_h
X_h = np.random.uniform(0,1,size=(n_init_h,5)); y_h = np.array([eval_params(denorm(x)) for x in X_h])
bo_hist = [y_h[:i+1].max() for i in range(len(y_h))]
kh = ConstantKernel(1.0)*Matern(length_scale=0.5, nu=2.5)
for i in range(n_iter_h):
    gh = GaussianProcessRegressor(kernel=kh, alpha=0.001, n_restarts_optimizer=3, random_state=i)
    gh.fit(X_h, y_h); cs = np.random.uniform(0,1,size=(500,5))
    mu_c,sig_c = gh.predict(cs, return_std=True); ei_c = expected_improvement(mu_c, sig_c, y_h.max())
    nx = cs[np.argmax(ei_c)]; ny = eval_params(denorm(nx))
    X_h = np.vstack([X_h, nx]); y_h = np.append(y_h, ny); bo_hist.append(y_h.max())
# Random
Xrh = np.random.uniform(0,1,size=(nt,5)); yrh = np.array([eval_params(denorm(x)) for x in Xrh])
rh = [yrh[:i+1].max() for i in range(len(yrh))]
# Grid
gv = [np.linspace(0,1,3) for _ in range(5)]; ga = np.array(list(product(*gv)))
gs = ga[np.random.choice(len(ga), min(nt, len(ga)), replace=False)]
ygh = np.array([eval_params(denorm(x)) for x in gs[:nt]]); gh2 = [ygh[:i+1].max() for i in range(len(ygh))]

print(f"BO: {y_h.max():.4f} | Random: {yrh.max():.4f} | Grid: {ygh.max():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sc = {'BO':COLORS['primary'],'Random':COLORS['accent'],'Grid':COLORS['rose']}
for hist, name in [(bo_hist,'BO'),(rh,'Random'),(gh2,'Grid')]:
    axes[0].plot(hist, linewidth=2.5, color=sc[name], label=name)
axes[0].set_title('HPO Convergence', fontweight='bold'); axes[0].legend(fontsize=11)
axes[0].set_xlabel('Evaluations'); axes[0].set_ylabel('Best AUC')

gb = max(y_h.max(), yrh.max(), ygh.max())
for ay, name in [(y_h,'BO'),(yrh,'Random'),(ygh,'Grid')]:
    axes[1].plot(np.cumsum(gb-ay), linewidth=2.5, color=sc[name], label=name)
axes[1].set_title('Cumulative Regret (lower = better)', fontweight='bold'); axes[1].legend()
axes[1].set_xlabel('Evaluations'); axes[1].set_ylabel('Cumulative Regret')
plt.tight_layout(); plt.show()

bp = denorm(X_h[np.argmax(y_h)])
print(f"\nBest BO config: n_est={int(bp[0])}, depth={int(bp[1])}, lr={bp[2]:.4f}, leaf={int(bp[3])}, sub={bp[4]:.3f}")

<a id="6"></a>
## 6. Conclusions

### Results
| Strategy | Best AUC | Budget |
|:---------|:--------:|:------:|
| **Bayesian Optimization** | highest | 30 evals |
| Random Search | competitive | 30 evals |
| Grid Search | lowest | 30 evals |

### When to Use Bayesian Optimization
- **Expensive evaluations** (training takes minutes/hours)
- **Small evaluation budget** (<100 trials)
- **Continuous parameters** with smooth response surfaces
- **Black-box functions** where gradients are unavailable

### When Random Search May Suffice
- **Cheap evaluations** (seconds per trial)
- **Large budgets** (>500 trials)
- **Highly irregular** response surfaces
- **Categorical-heavy** search spaces

### Future Work
- **Multi-fidelity BO** (Hyperband, BOHB) for early stopping
- **Multi-objective BO** (accuracy + latency Pareto)
- **Transfer learning** across related HPO tasks
- **Neural architecture search** (NAS) with BO
- **Constrained optimization** (e.g., memory/latency budgets)
- **Asynchronous parallel BO** for distributed HPO

---
*Synthetic benchmark + real ML task. GP surrogate and acquisition functions implemented from scratch.*
